# Module 2: Incident Counts Are Not Gaussian

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Almost every model in the Intermediate series treated a monthly count, or a
rate built from one, as though it were a continuous quantity with constant
spread. That is the default in most software and it is wrong in three specific
ways for public safety data.

This module names the three, shows what each one costs, and replaces the
default with the model that belongs.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

import statsmodels.api as sm
import statsmodels.formula.api as smf

f = final[final["year_month"] <= "2025-12"].copy()
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
idx = pd.PeriodIndex(f["year_month"], freq="M")
f["t"] = (idx.year - 2019) * 12 + idx.month - 1
f["mon"] = idx.month
per_year = lambda b: 100 * (np.exp(12 * b) - 1)
print(f"{len(f)} agency months")

## 2. What ordinary regression assumes, and why counts break it

| Assumption | What counts do instead |
|---|---|
| the outcome can take any value | a count cannot be negative, and cannot be 2.4 |
| the spread is the same everywhere | the spread of a count grows with its level |
| the errors are normal | a count averaging 0.8 a month is nothing like normal |

The middle one is the most consequential and the least noticed. For events
that occur roughly independently, the **variance of a count is close to its
mean**. A month expecting 100 incidents varies by about 10; a month expecting
1 varies by about 1. Ordinary regression assumes a single spread for both.

## 3. Measure the dispersion properly

The temptation is to divide the variance by the mean and look for 1. Do not:
that ratio is inflated by every bit of trend and season in the series, and the
inflation grows with agency size because larger agencies have more visible
structure.

Fit the model first, then measure what is left.

In [ ]:
rows = []
for aid, g in f.groupby("agency_id"):
    poi = smf.glm("n_uof ~ t + C(mon)", data=g, family=sm.families.Poisson(),
                  offset=np.log(g["n_arrests"])).fit()
    rows.append({
        "agency": (g["agency_name"].iloc[0].replace(" Police Department", "")
                   .replace(" Sheriff's Office", "").replace(" Police", "")),
        "mean a month": round(g["n_uof"].mean(), 1),
        "raw variance over mean": round(g["n_uof"].var() / g["n_uof"].mean(), 2),
        "dispersion after a model": round((poi.resid_pearson ** 2).sum() / poi.df_resid, 2),
    })
pd.DataFrame(rows).sort_values("mean a month").set_index("agency")

The two columns tell completely different stories.

**Raw, Ashfell looks eight times overdispersed.** After a model with a trend
and month terms, it is 1.76. Stonewick goes from 4.34 to 1.04, and Summit
County from 3.42 to 1.12. **Most of the apparent overdispersion was the
seasonal pattern and the trend**, not extra randomness.

Several agencies come out slightly **below** 1, which is sampling noise in the
dispersion estimate rather than genuine underdispersion.

The practical rule: dispersion is a property of a model's residuals, not of a
series. Reporting a raw variance to mean ratio overstates it, and overstates it
worse for the agencies with the most structure.

## 4. Exposure belongs in the model, not in the outcome

Intermediate [Module 3](../../Intermediate/Module_03_Choosing_A_Denominator.md)
divided incidents by arrests to get a rate, then modelled the rate. There is a
better way.

Model the **count**, and put the log of the exposure in as an **offset**: a
term with its coefficient fixed at 1. That says an agency with twice the
arrests is expected to have twice the incidents, all else equal, which is
exactly what a rate means, while keeping the outcome a count so that the
variance assumption stays right.

In [ ]:
g = f[f["agency_id"] == "A012"].copy()

poisson = smf.glm("n_uof ~ t + C(mon)", data=g, family=sm.families.Poisson(),
                  offset=np.log(g["n_arrests"])).fit()

print(f"trend: {per_year(poisson.params['t']):+.2f} percent a year")
print("the coefficient on t is already a proportional change, because the model is on the log scale")

## 5. Poisson, and then the honest version

Poisson assumes the dispersion is exactly 1. Ashfell's is 1.76, so Poisson's
standard errors are too small. The negative binomial adds one parameter for
the extra spread and fixes them.

In [ ]:
aux = ((g["n_uof"] - poisson.mu) ** 2 - g["n_uof"]) / poisson.mu
alpha = sm.OLS(aux, poisson.mu).fit().params.iloc[0]

negbin = smf.glm("n_uof ~ t + C(mon)", data=g,
                 family=sm.families.NegativeBinomial(alpha=alpha),
                 offset=np.log(g["n_arrests"])).fit()

print(f"estimated extra spread, alpha = {alpha:.4f}\n")
for name, mod in [("Poisson", poisson), ("negative binomial", negbin)]:
    lo, hi = mod.conf_int().loc["t"]
    print(f"  {name:20s} {per_year(mod.params['t']):+.2f} percent a year  "
          f"[{per_year(lo):+.2f}, {per_year(hi):+.2f}]  "
          f"width {per_year(hi) - per_year(lo):.2f}   "
          f"deviance over df {mod.deviance / mod.df_resid:.2f}")
print("\n  the truth built into the data: -4.88 percent a year")

**The estimate barely moves. The interval widens by about a fifth.**

That is the whole of it. Overdispersion does not bias the slope; it makes you
more confident than you are entitled to be. A Poisson model fitted to
overdispersed counts will report intervals that are too narrow and p values
that are too small, and nothing in its output will look wrong.

The deviance over degrees of freedom column is the quick check. Around 1 means
Poisson is adequate. Ashfell's 1.78 says it is not.

## 6. Where the Gaussian assumption fails visibly

At a large agency the wrong model gives slightly wrong intervals. At a small
one it gives impossible ones.

In [ ]:
from scipy import stats

e = f[f["agency_id"] == "A006"].copy()          # Orrindale, eight officers

ols = smf.ols("n_uof ~ t + C(mon)", data=e).fit()
band = ols.get_prediction(e).summary_frame(alpha=0.05)

poi_small = smf.glm("n_uof ~ t + C(mon)", data=e, family=sm.families.Poisson(),
                    offset=np.log(e["n_arrests"])).fit()
poi_lower = stats.poisson.ppf(0.025, poi_small.mu)

print(f"ordinary regression, 95 percent prediction intervals")
print(f"  months with a negative lower bound: {int((band['obs_ci_lower'] < 0).sum())} of {len(e)}")
print(f"  lowest lower bound: {band['obs_ci_lower'].min():.2f}")
print(f"  a typical interval: [{band['obs_ci_lower'].iloc[0]:.2f}, "
      f"{band['obs_ci_upper'].iloc[0]:.2f}]")
print(f"\nPoisson with exposure")
print(f"  months with a negative lower bound: {int((poi_lower < 0).sum())} of {len(e)}")

**Every single one of Orrindale's 84 ordinary prediction intervals extends
below zero**, for a quantity that cannot be negative. The Poisson intervals
never do, because the model knows what a count is.

Note what does **not** go wrong: the fitted values themselves stay positive.
The failure is in the intervals, which is the part people are least likely to
plot and most likely to quote.

In [ ]:
low = ols.resid[ols.fittedvalues < ols.fittedvalues.median()]
high = ols.resid[ols.fittedvalues >= ols.fittedvalues.median()]
print(f"ordinary regression residual variance")
print(f"  where the fitted value is low : {low.var():.2f}")
print(f"  where the fitted value is high: {high.var():.2f}")
print("\nthe spread grows with the level, exactly as a count model expects")
print("and exactly what ordinary regression assumes does not happen")

## 7. Which model to reach for

| Situation | Model |
|---|---|
| counts, dispersion close to 1 | Poisson with `offset=np.log(exposure)` |
| counts, dispersion above about 1.2 | negative binomial |
| many months at zero | see [Module 9](Module_09_Rare_Events.ipynb) |
| the quantity is genuinely continuous | ordinary regression is fine |
| you only need the point estimate and the series is large | the difference is small, but say so |

The last row is the honest caveat. For Ashfell the Poisson and negative
binomial slopes agree to two decimal places. **If all you report is the slope,
the choice hardly matters. The moment you report an interval, a p value, or a
prediction, it does.**

## Exercise

Fit Poisson and negative binomial to Tarnbridge, which Intermediate
[Module 8](../../Intermediate/Module_08_Rolling_Statistics_And_Control_Limits.md)
found to have a dispersion of about 1.6 using a different method. Do the two
approaches agree?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    g2 = f[f["agency_id"] == AGENCY].copy()
    p2 = smf.glm("n_uof ~ t + C(mon)", data=g2, family=sm.families.Poisson(),
                 offset=np.log(g2["n_arrests"])).fit()
    phi = (p2.resid_pearson ** 2).sum() / p2.df_resid
    aux2 = ((g2["n_uof"] - p2.mu) ** 2 - g2["n_uof"]) / p2.mu
    a2 = sm.OLS(aux2, p2.mu).fit().params.iloc[0]
    n2 = smf.glm("n_uof ~ t + C(mon)", data=g2,
                 family=sm.families.NegativeBinomial(alpha=a2),
                 offset=np.log(g2["n_arrests"])).fit()
    print(f"Pearson dispersion: {phi:.2f}")
    for name, mod in [("Poisson", p2), ("negative binomial", n2)]:
        lo, hi = mod.conf_int().loc["t"]
        print(f"  {name:20s} {per_year(mod.params['t']):+.2f}  "
              f"[{per_year(lo):+.2f}, {per_year(hi):+.2f}]  "
              f"width {per_year(hi) - per_year(lo):.2f}")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

The Pearson dispersion comes out at **1.38**, against the **1.59** that
Intermediate Module 8 obtained. Both say the same thing qualitatively, that
Poisson would be too tight, and they are not the same number.

The gap is informative rather than troubling. Module 8 compared the counts
against an expectation built from an STL decomposition of the counts alone.
This model compares them against a regression that also carries **arrests as
exposure**, so some of the month to month movement that Module 8 was counting
as unexplained spread is here explained by how busy the agency was.

**Dispersion is a property of a model, not of a series.** A model that
explains more leaves less of it. That is the same point section 3 made about
raw variance over mean, arriving from the other direction, and it means a
reported dispersion figure is meaningless without the model it came from.

One thing the output shows in passing: the estimated trend is about minus 8
percent a year, well below the minus 4.88 built into the data. Tarnbridge
adopted the de escalation programme in July 2023, so a single straight trend
across the whole window is averaging two different regimes. Fitting that
properly is [Module 11](Module_11_Interrupted_Time_Series.ipynb).

</details>

---

**Next:** [Module 3, Model Selection, Diagnostics and Honest Uncertainty](Module_03_Model_Selection_And_Uncertainty.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*